# Visual decoding
Classification and retrieval panels from release `test_scores.npz` files.
Each point is one participant/animal; bars show means. No runs or trials are counted
as additional participants. Scores are recomputed from the saved galleries.
The similarity matrix is descriptive. Image-generation models are outside this core release.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Run from this notebook directory or the release root.
ROOT = Path.cwd() if (Path.cwd() / "brainae").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "notebooks"))
from figure_utils import *
style()
OUTPUT = ROOT / "figures"


In [ ]:
# Add all participant/animal run directories you want to report; one entry per recording.
RUNS = {'eeg': [ROOT / 'outputs/eeg/sub-01'], 'meg': [], 'mua': []}


In [ ]:
records=[]
for modality, runs in RUNS.items():
    for run in runs:
        with np.load(run / 'test_scores.npz', allow_pickle=False) as z:
            labels=z['labels']
            for task in ['classification','retrieval']:
                scores=z[task]
                assert scores.ndim==2 and len(scores)==len(labels)
                ranking=np.argsort(-scores,axis=1,kind='stable')
                for k in [1,3,5]:
                    records.append({'modality':modality,'recording':run.name,'task':task,'k':k,
                        'accuracy':np.mean((ranking[:,:min(k,scores.shape[1])]==labels[:,None]).any(1)),
                        'gallery':scores.shape[1]})
results=pd.DataFrame(records)
assert len(results), 'Configure at least one completed run.'
OUTPUT.mkdir(parents=True,exist_ok=True)
results.to_csv(OUTPUT/'decoding_scores.csv',index=False)
results


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(7,3),layout='constrained')
active=[m for m in MODALITIES if RUNS[m]]
for ax,task,letter in zip(axes,['classification','retrieval'],['a','b']):
    for j,k in enumerate([1,3,5]):
        for i,modality in enumerate(active):
            part=results.query('modality==@modality and task==@task and k==@k')
            values=100*part.accuracy.to_numpy(); x=i+(j-1)*.24
            ax.bar(x,values.mean(),width=.21,color=[BLUE,RED,GRAY][j],alpha=.7,
                   label=f'Top-{k}' if i==0 else None)
            offsets=np.linspace(-.055,.055,len(values)) if len(values)>1 else np.zeros(1)
            ax.scatter(x+offsets,values,c='black',s=15,zorder=3,marker=['o','s','^'][j])
    ax.set(xticks=range(len(active)),xticklabels=[m.upper() for m in active],
           ylabel='Accuracy (%)',title=task.capitalize(),ylim=(0,100))
    panel(ax,letter)
axes[0].legend(ncol=3,fontsize=8)
save(fig,OUTPUT,'decoding_performance')

run=next(run for runs in RUNS.values() for run in runs)
with np.load(run/'test_scores.npz',allow_pickle=False) as z:
    embeddings=z['embeddings']; scores=z['classification']
embeddings=embeddings/np.maximum(np.linalg.norm(embeddings,axis=1,keepdims=True),1e-12)
fig,axes=plt.subplots(1,2,figsize=(6.6,3),layout='constrained')
for ax,matrix,title in zip(axes,[scores,embeddings@embeddings.T],['Classification scores','Neural representation similarity']):
    im=ax.imshow(matrix,cmap='RdBu_r',aspect='equal',interpolation='nearest')
    ax.set(title=title,xlabel='Gallery image / stimulus',ylabel='Query image / stimulus')
    fig.colorbar(im,ax=ax,shrink=.7)
save(fig,OUTPUT,'decoding_similarity')
